# Final submissions — Kaggle "perfect-fit"

This notebook is self-contained: given `../data/dataset.csv` and
`../data/test.csv`, it produces the four Kaggle submissions we kept at the
end of the competition.

| Output CSV | CV MAE | Public LB MAE | Description |
|---|---|---|---|
| `submission_ebm_heavy_smooth.csv` | 3.08 | **4.90** | Single EBM with heavy smoothing. Cleanest "one model" baseline. |
| `submission_ensemble_cross_LE.csv` | 2.97 | **2.94** | 0.5·LIN(no x9) + 0.5·EBM(no x4). Our best LB-confirmed submission. |
| `submission_ensemble_triple_locked_b_lambda50.csv` | 2.82 | untested | 0.25·LIN_b + 0.25·EBM(no x4) + 0.5·EBM(full). Integer-locked linear. |
| `submission_router_A1_triple.csv` | **1.84** | untested | Route "safe" rows to A1 closed form, others to the triple ensemble. |

`CLAUDE.md` has the full narrative. `LEARNINGS.md` distils the portable
lessons.

**Runtime**: ~15 min end-to-end on a laptop. The 5-fold CV cell is the
slowest; you can skip it and jump straight to full-data training if
you trust the numbers above.


## 1. Setup


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from interpret.glassbox import ExplainableBoostingRegressor

SEED = 42
SENTINEL = 999.0  # the magic missing-value code on x5
N_SPLITS = 5

# Try both notebook-run-from-notebooks/ and notebook-run-from-repo-root:
HERE = Path.cwd()
REPO = HERE.parent if (HERE / "../data/dataset.csv").exists() else HERE
DATA = REPO / "data"
SUBS = REPO / "submissions"
SUBS.mkdir(exist_ok=True)

FEATURES_ALL = ["x1", "x2", "x4", "x5", "x8", "x9", "x10", "x11"]


## 2. Load data


In [2]:
train = pd.read_csv(DATA / "dataset.csv").reset_index(drop=True)
test  = pd.read_csv(DATA / "test.csv").reset_index(drop=True)
y     = train["target"].values
is_sent = (train["x5"] == SENTINEL).to_numpy()
print(f"train: {len(train)} rows  sentinels={is_sent.sum()}")
print(f"test : {len(test)} rows  sentinels={(test['x5']==SENTINEL).sum()}")


train: 1500 rows  sentinels=222
test : 1500 rows  sentinels=228


## 3. Shared preprocessing

Three helpers used across every model:

- **`preprocess(df, features, x5_median)`** — builds the EBM feature frame
  (x5 imputed with training median, adds `x5_is_sentinel` and binary
  `city` columns).
- **`design_matrix(df, x5_median, include_x4, include_x9)`** — builds
  the hand-crafted linear basis with `x1²`, `cos(5π·x2)`, `x10·x11`,
  and toggles for x4/x9 (used for the `LIN_x4` view).
- **`safe_mask(df)`** — the routing predicate. "Safe" means: x5 is not
  a sentinel, x4 is outside the training gap, and x9 matches its
  x4-cluster (true invariant of the DGP).


In [3]:
def preprocess(df: pd.DataFrame, features: list[str], x5_median: float) -> pd.DataFrame:
    out = df[features].copy()
    out["x5"] = out["x5"].where(out["x5"] != SENTINEL, x5_median)
    out["x5_is_sentinel"] = (df["x5"] == SENTINEL).astype(float)
    out["city"] = (df["City"] == "Zaragoza").astype(float)
    return out


def ebm_features(with_x4: bool, with_x9: bool) -> list[str]:
    feats = list(FEATURES_ALL)
    if not with_x4: feats.remove("x4")
    if not with_x9: feats.remove("x9")
    return feats


def design_matrix(df: pd.DataFrame, x5_median: float,
                  include_x4: bool, include_x9: bool) -> np.ndarray:
    is_sent = (df["x5"] == SENTINEL).astype(float).values
    x5 = df["x5"].where(df["x5"] != SENTINEL, x5_median).values
    city = (df["City"] == "Zaragoza").astype(float).values
    cols = [
        df["x1"].values ** 2,
        np.cos(5 * np.pi * df["x2"].values),
    ]
    if include_x4: cols.append(df["x4"].values)
    cols += [x5, is_sent, df["x8"].values, df["x10"].values, df["x11"].values,
             df["x10"].values * df["x11"].values, city]
    if include_x9: cols.append(df["x9"].values)
    return np.column_stack(cols)


X4_GAP_THRESH = 0.167  # empirical bimodal gap in x4


def safe_mask(df: pd.DataFrame) -> np.ndarray:
    x4 = df["x4"].to_numpy()
    x9 = df["x9"].to_numpy()
    sent = (df["x5"].to_numpy() == SENTINEL)
    x4_clear = np.abs(x4) > X4_GAP_THRESH
    cluster_match = ((x4 > 0) & (x9 > 5)) | ((x4 < 0) & (x9 < 5))
    return (~sent) & x4_clear & cluster_match


## 4. Model builders

Two EBM configurations:

- **`fit_ebm_heavy_smooth`** — `smoothing_rounds=2000, interaction_smoothing_rounds=500,
  max_rounds=2000`. Used for the single-model `ebm_heavy_smooth` submission
  (LB 4.90).
- **`fit_ebm_4k`** — `smoothing_rounds=4000, interaction_smoothing_rounds=1000,
  max_rounds=4000`. Used for the ensembles (`cross_LE`, triple, router).
  Slightly stronger smoothing → better CV (3.03 vs 3.08) at 2× training time.

The linear view uses `LinearRegression` on `design_matrix(...)` either
free-fit or pinned to the **integer-locked** coefficient set B that
matches the reverse-engineered A1/A2 formulas. The intercept is always
free so that the imputed-sentinel rows receive a correction.

`approach1_predict` is the A1 closed form — zero free parameters,
near-perfectly interpolates training (CV 1.80) but catastrophically
fails on the test set's x4-gap rows (LB 10.80). In the router we only
apply it where we have strong prior evidence it will work.


In [4]:
EBM_KW_HEAVY_SMOOTH = dict(
    interactions=10, max_rounds=2000, min_samples_leaf=10, max_bins=128,
    smoothing_rounds=2000, interaction_smoothing_rounds=500, random_state=SEED,
)
EBM_KW_4K = dict(
    interactions=10, max_rounds=4000, min_samples_leaf=10, max_bins=128,
    smoothing_rounds=4000, interaction_smoothing_rounds=1000, random_state=SEED,
)

def fit_ebm_heavy_smooth(X, y):
    return ExplainableBoostingRegressor(**EBM_KW_HEAVY_SMOOTH).fit(X, y)

def fit_ebm_4k(X, y):
    return ExplainableBoostingRegressor(**EBM_KW_4K).fit(X, y)


# Integer-locked coefs for LIN_x4 (A1/A2 declared integers).
# Column order must match design_matrix(include_x4=True, include_x9=False).
LOCKED_COEFS_B = {
    "x1^2": -100, "cos(5pi*x2)": +10, "x4": +30, "x5_imp": -8,
    "x5_is_sent": 0, "x8": +14, "x10": 0, "x11": 0,
    "x10*x11": +1, "city": -25,
}
LIN_COL_ORDER = ["x1^2", "cos(5pi*x2)", "x4", "x5_imp", "x5_is_sent",
                 "x8", "x10", "x11", "x10*x11", "city"]


def lin_x4_free(df_tr, df_va, x5_median):
    X_tr = design_matrix(df_tr, x5_median, include_x4=True, include_x9=False)
    X_va = design_matrix(df_va, x5_median, include_x4=True, include_x9=False)
    return LinearRegression().fit(X_tr, df_tr["target"].values).predict(X_va)


def lin_x4_locked_b(df_tr, df_va, x5_median):
    X_tr = design_matrix(df_tr, x5_median, include_x4=True, include_x9=False)
    X_va = design_matrix(df_va, x5_median, include_x4=True, include_x9=False)
    v = np.array([LOCKED_COEFS_B[c] for c in LIN_COL_ORDER])
    intercept = (df_tr["target"].values - X_tr @ v).mean()
    return X_va @ v + intercept


def approach1_predict(df: pd.DataFrame, x5_median: float) -> np.ndarray:
    """A1 closed form — reverse-engineered by a sibling branch.

    target = -100*x1^2 + 10*cos(5π*x2) + 15*x4 - 8*x5 + 15*x8
           - 4*x9 + x10*x11 - 25*is_zaragoza + 20*1(x4>0) + 92.5
    """
    x5 = df["x5"].where(df["x5"] != SENTINEL, x5_median).values
    is_zar = (df["City"] == "Zaragoza").astype(float).values
    x4_pos = (df["x4"].values > 0).astype(float)
    return (
        -100 * df["x1"].values ** 2
        + 10 * np.cos(5 * np.pi * df["x2"].values)
        + 15 * df["x4"].values
        - 8 * x5
        + 15 * df["x8"].values
        - 4 * df["x9"].values
        + df["x10"].values * df["x11"].values
        - 25 * is_zar
        + 20 * x4_pos
        + 92.5
    )


def mae(p, y):
    return float(np.mean(np.abs(p - y)))


## 5. 5-fold CV (optional — slow, ~8-10 min)

Reproduces the CV numbers quoted in the overview table. Set
`RUN_CV = False` to skip and jump straight to full-data training.


In [5]:
RUN_CV = True

if RUN_CV:
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    names = ["a1", "lin_x4_free", "lin_x4_b", "ebm_x9", "ebm_full", "ebm_heavy"]
    oof = {n: np.zeros(len(train)) for n in names}

    for fold, (tr, va) in enumerate(kf.split(train)):
        sub_tr = train.iloc[tr].reset_index(drop=True)
        sub_va = train.iloc[va].reset_index(drop=True)
        x5m = float(sub_tr.loc[sub_tr["x5"] != SENTINEL, "x5"].median())

        oof["a1"][va]          = approach1_predict(sub_va, x5m)
        oof["lin_x4_free"][va] = lin_x4_free(sub_tr, sub_va, x5m)
        oof["lin_x4_b"][va]    = lin_x4_locked_b(sub_tr, sub_va, x5m)

        feats = ebm_features(with_x4=False, with_x9=True)
        X_tr, X_va = preprocess(sub_tr, feats, x5m), preprocess(sub_va, feats, x5m)
        oof["ebm_x9"][va] = fit_ebm_4k(X_tr, sub_tr["target"].values).predict(X_va)

        feats = ebm_features(with_x4=True, with_x9=True)
        X_tr, X_va = preprocess(sub_tr, feats, x5m), preprocess(sub_va, feats, x5m)
        oof["ebm_full"][va] = fit_ebm_4k(X_tr, sub_tr["target"].values).predict(X_va)

        X_tr, X_va = preprocess(sub_tr, FEATURES_ALL, x5m), preprocess(sub_va, FEATURES_ALL, x5m)
        oof["ebm_heavy"][va] = fit_ebm_heavy_smooth(X_tr, sub_tr["target"].values).predict(X_va)

        print(f"  fold {fold+1}/{N_SPLITS} done")

    # Report
    triple  = 0.25 * oof["lin_x4_b"] + 0.25 * oof["ebm_x9"] + 0.50 * oof["ebm_full"]
    crossLE = 0.50 * oof["lin_x4_free"] + 0.50 * oof["ebm_x9"]
    routed  = np.where(safe_mask(train), oof["a1"], triple)

    print("\n{:<32s} {:>8s} {:>10s}".format("model", "overall", "non-sent"))
    print("-" * 54)
    for label, p in [
        ("A1 closed form",               oof["a1"]),
        ("LIN_x4 free",                  oof["lin_x4_free"]),
        ("LIN_x4 locked_b",              oof["lin_x4_b"]),
        ("EBM(no x4)",                   oof["ebm_x9"]),
        ("EBM(full)",                    oof["ebm_full"]),
        ("EBM heavy_smooth",             oof["ebm_heavy"]),
        ("cross_LE 0.5*(LIN_free+EBM9)", crossLE),
        ("triple 0.25+0.25+0.5",         triple),
        ("router(safe->A1, else->triple)", routed),
    ]:
        print(f"{label:<32s} {mae(p, y):8.3f} {mae(p[~is_sent], y[~is_sent]):10.3f}")


  fold 1/5 done


  fold 2/5 done


  fold 3/5 done


  fold 4/5 done


  fold 5/5 done

model                             overall   non-sent
------------------------------------------------------
A1 closed form                      1.800      0.376
LIN_x4 free                         3.695      2.537
LIN_x4 locked_b                     3.672      2.530
EBM(no x4)                          3.727      2.508
EBM(full)                           3.030      1.735
EBM heavy_smooth                    3.081      1.792
cross_LE 0.5*(LIN_free+EBM9)        2.973      1.718
triple 0.25+0.25+0.5                2.824      1.536
router(safe->A1, else->triple)      1.839      0.380


## 6. Full-data training

Every base model is refit on the full 1,500-row `dataset.csv`, and its
test predictions are cached in `preds`. This is the only step the
submission builders depend on — the CV section above is informational.


In [6]:
x5m = float(train.loc[train["x5"] != SENTINEL, "x5"].median())
preds: dict[str, np.ndarray] = {}

# A1 closed form
preds["a1"] = approach1_predict(test, x5m)

# Linear views — free-fit and integer-locked
X_tr = design_matrix(train, x5m, include_x4=True, include_x9=False)
X_te = design_matrix(test,  x5m, include_x4=True, include_x9=False)
preds["lin_x4_free"] = LinearRegression().fit(X_tr, y).predict(X_te)
v = np.array([LOCKED_COEFS_B[c] for c in LIN_COL_ORDER])
preds["lin_x4_b"] = X_te @ v + (y - X_tr @ v).mean()

# EBM views
feats = ebm_features(with_x4=False, with_x9=True)
preds["ebm_x9"] = fit_ebm_4k(
    preprocess(train, feats, x5m), y).predict(preprocess(test, feats, x5m))

feats = ebm_features(with_x4=True, with_x9=True)
preds["ebm_full"] = fit_ebm_4k(
    preprocess(train, feats, x5m), y).predict(preprocess(test, feats, x5m))

preds["ebm_heavy"] = fit_ebm_heavy_smooth(
    preprocess(train, FEATURES_ALL, x5m), y).predict(preprocess(test, FEATURES_ALL, x5m))

for k, p in preds.items():
    print(f"  {k:<14s}  mean={p.mean():+.3f}  std={p.std():.3f}  "
          f"range=[{p.min():+.2f}, {p.max():+.2f}]")


  a1              mean=-3.871  std=26.069  range=[-87.32, +71.95]
  lin_x4_free     mean=-4.022  std=23.204  range=[-79.07, +63.07]
  lin_x4_b        mean=-4.015  std=23.222  range=[-79.05, +62.89]
  ebm_x9          mean=-4.076  std=24.247  range=[-77.27, +75.15]
  ebm_full        mean=-3.979  std=22.957  range=[-77.61, +69.12]
  ebm_heavy       mean=-3.832  std=22.989  range=[-77.93, +69.82]


## 7. Build the four submissions

Each submission is a simple arithmetic combination of the cached `preds`.
See the formulas in the overview table at the top.


In [7]:
triple_b = 0.25 * preds["lin_x4_b"] + 0.25 * preds["ebm_x9"] + 0.50 * preds["ebm_full"]
cross_LE = 0.50 * preds["lin_x4_free"] + 0.50 * preds["ebm_x9"]
routed   = np.where(safe_mask(test), preds["a1"], triple_b)

outputs = {
    "submission_ebm_heavy_smooth.csv":                        preds["ebm_heavy"],
    "submission_ensemble_cross_LE.csv":                       cross_LE,
    "submission_ensemble_triple_locked_b_lambda50.csv":       triple_b,
    "submission_router_A1_triple.csv":                        routed,
}

for name, p in outputs.items():
    pd.DataFrame({"id": test["id"], "target": p}).to_csv(SUBS / name, index=False)
    print(f"  wrote {name}  mean={p.mean():+.3f}  range=[{p.min():+.2f}, {p.max():+.2f}]")


  wrote submission_ebm_heavy_smooth.csv  mean=-3.832  range=[-77.93, +69.82]
  wrote submission_ensemble_cross_LE.csv  mean=-4.049  range=[-76.36, +66.52]
  wrote submission_ensemble_triple_locked_b_lambda50.csv  mean=-4.012  range=[-76.99, +67.88]
  wrote submission_router_A1_triple.csv  mean=-4.061  range=[-80.30, +67.88]


## 8. Sanity checks

Validate each submission has the right shape, no NaNs, and a plausible
target distribution. If the existing Kaggle-confirmed CSVs are still
present, diff against them — we expect near-identical numbers for the
deterministic paths (`ebm_heavy_smooth`, `cross_LE`, `router`).


In [8]:
n_expected = len(test)
for name in outputs:
    df = pd.read_csv(SUBS / name)
    assert list(df.columns) == ["id", "target"], f"{name}: bad columns {df.columns}"
    assert len(df) == n_expected, f"{name}: length {len(df)} != {n_expected}"
    assert df["target"].notna().all(), f"{name}: NaN predictions"
    assert np.isfinite(df["target"]).all(), f"{name}: non-finite predictions"
    assert (df["id"].values == test["id"].values).all(), f"{name}: id mismatch"
    print(f"  {name}: OK  n={len(df)}  mean={df['target'].mean():+.3f}")


  submission_ebm_heavy_smooth.csv: OK  n=1500  mean=-3.832
  submission_ensemble_cross_LE.csv: OK  n=1500  mean=-4.049
  submission_ensemble_triple_locked_b_lambda50.csv: OK  n=1500  mean=-4.012
  submission_router_A1_triple.csv: OK  n=1500  mean=-4.061


## 9. What I would try next

See `LEARNINGS.md` for portable patterns (cross-view ensembling, router
pattern, sentinel-noise floor arithmetic). Two open threads in `CLAUDE.md`:

- **Hidden clamp trigger**. Inside the `x4<0 ∧ x8<0` quadrant, 23% of
  training rows have `residual ≈ -18.4·x8` relative to A1. A LightGBM
  classifier reaches AUC 0.76 but that isn't sharp enough to route on
  — false positives cost more than true positives gain. Better features
  (beyond x6/x7 angle, which is null) or a different parameterisation
  might push AUC past the 0.9+ threshold needed.
- **DGP oracle**. A1 + the clamp rule would get the top-4 LB cluster
  (1.65–1.71), but only if we can generate targets for off-diagonal
  (x4, x9) test rows — which requires the true DGP. Synthetic
  augmentation using A1 backfires because A1's x9 coefficient is itself
  wrong on test.
